In [ ]:
import config
from huggingface_hub import InferenceClient

MODELS = getattr(
    config,
    "HF_MODELS",
    ["meta-llama/Llama-3.1-8B-Instruct"],
)

def generate_response(prompt: str, temperature: float = 0.3, max_tokens: int = 512) -> str:
    key = getattr(config, "HF_API_KEY", None)
    if not key:
        return "Error: HF_API_KEY missing in config.py"

    last_err = None
    for m in MODELS:
        try:
            c = InferenceClient(model=m, token=key)
            r = c.chat_completion(
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return r.choices[0].message.content
        except Exception as e:
            last_err = e

    return (
        "Hugging Face model failed.\n"
        f"Tried models: {MODELS}\n"
        "Fix:\n"
        "1) Switch to Groq by importing groq.py in main.py OR\n"
        "2) Replace HF model in hf.py (HF_MODELS).\n"
        f"Details: {type(last_err).__name__}: {last_err}"
    )


In [ ]:
# main.py
from io import BytesIO
import requests
import streamlit as st
from huggingface_hub import InferenceClient

import config

MODEL_ID = "stabilityai/stable-diffusion-3-medium-diffusers"
FILTER_API_URL = "https://filters-zeta.vercel.app/api/filter"

ENHANCE_SYS = (
    "Improve prompts for text-to-image. Return ONLY the enhanced prompt. "
    "Add subject, style, lighting, camera angle, background, colors. Keep it safe."
)

# This is only for image quality guidance, not safety filtering
NEGATIVE = "low quality, blurry, distorted, watermark, text, cropped"

img_client = InferenceClient(provider="hf-inference", api_key=config.HF_API_KEY)


def check_prompt_with_filter_api(prompt: str):
    try:
        response = requests.post(
            FILTER_API_URL,
            json={"prompt": prompt},
            timeout=10,
        )
        response.raise_for_status()
        data = response.json()

        if not isinstance(data, dict):
            return {"ok": False, "reason": "Invalid filter API response."}

        return data

    except Exception as e:
        return {
            "ok": False,
            "reason": f"Filter API error: {str(e)}",
        }


def enhance_prompt(raw: str) -> str:
    from hf import generate_response

    out = generate_response(
        f"{ENHANCE_SYS}\nUser prompt: {raw}",
        temperature=0.4,
        max_tokens=220,
    )
    return (out or raw).strip()


def gen_image(prompt: str):
    filter_result = check_prompt_with_filter_api(prompt)
    if not filter_result.get("ok"):
        return None, f"⚠️ Prompt blocked by safety filter. {filter_result.get('reason', 'Unsafe prompt')}"

    try:
        return img_client.text_to_image(
            prompt=prompt,
            negative_prompt=NEGATIVE,
            model=MODEL_ID,
        ), None

    except Exception as e:
        msg = str(e)

        if "negative_prompt" in msg or "unexpected keyword" in msg:
            try:
                return img_client.text_to_image(
                    prompt=prompt,
                    model=MODEL_ID,
                ), None
            except Exception as e2:
                msg = str(e2)

        if any(x in msg for x in ["402", "Payment Required", "pre-paid credits"]):
            return None, "❌ Image backend requires credits or model not available on hf-inference.\n\nRaw error: " + msg

        if "404" in msg or "Not Found" in msg:
            return None, "❌ Model not served on this provider route (hf-inference).\n\nRaw error: " + msg

        return None, "Error during image generation: " + msg


def main():
    st.set_page_config(page_title="Safe AI Image Generator", layout="centered")
    st.title("🖼️ Safe AI Image Generator")
    st.info("Flow: Enter a prompt → enhance it → check it using the deployed safety API → generate the image.")

    with st.form("image_form"):
        raw = st.text_area(
            "Image Description",
            height=120,
            placeholder="Example: A cozy cabin in snowy mountains at sunrise, cinematic lighting",
        )
        submit = st.form_submit_button("Generate Image")

    if submit:
        raw = raw.strip()

        if not raw:
            st.warning("⚠️ Please enter an image description.")
            return

        raw_check = check_prompt_with_filter_api(raw)
        if not raw_check.get("ok"):
            st.error(f"⚠️ Prompt blocked. {raw_check.get('reason', 'Unsafe prompt')}")
            return

        with st.spinner("Enhancing your prompt..."):
            final_prompt = enhance_prompt(raw)

        enhanced_check = check_prompt_with_filter_api(final_prompt)
        if not enhanced_check.get("ok"):
            st.error(f"⚠️ Enhanced prompt blocked. {enhanced_check.get('reason', 'Unsafe prompt')}")
            return

        st.markdown("#### Enhanced Prompt")
        st.code(final_prompt)

        with st.spinner("Generating image..."):
            img, err = gen_image(final_prompt)

        if err:
            st.error(err)
            return

        st.image(img, caption="Generated Image", use_container_width=True)
        st.session_state.generated_image = img

    img = st.session_state.get("generated_image")
    if img:
        buf = BytesIO()
        img.save(buf, format="PNG")
        st.download_button(
            "📥 Download Image",
            buf.getvalue(),
            "ai_generated_image.png",
            "image/png",
        )


if __name__ == "__main__":
    main()

In [ ]:
from io import BytesIO

import requests
import streamlit as st
from huggingface_hub import InferenceClient

import config
from hf import generate_response


# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

MODEL_ID = "stabilityai/stable-diffusion-3-medium-diffusers"

FILTER_API_URL = "https://filters-zeta.vercel.app/api/filter"

ENHANCE_SYS = """
Improve the user's text-to-image prompt.

Return ONLY the improved prompt.

Add useful visual details such as:
- subject
- environment
- artistic style
- lighting
- camera angle
- background
- colors
- composition

Do not introduce unsafe, violent, sexual, hateful,
illegal, or otherwise harmful content.
"""

# Negative prompt improves image quality.
# It is NOT used as the safety filter.
NEGATIVE_PROMPT = (
    "low quality, blurry, distorted, deformed, "
    "watermark, text, cropped, poorly drawn"
)


# ---------------------------------------------------------
# Hugging Face Image Client
# ---------------------------------------------------------

image_client = InferenceClient(
    provider="hf-inference",
    api_key=config.HF_API_KEY,
)


# ---------------------------------------------------------
# Prompt Safety Filter
# ---------------------------------------------------------

def check_prompt_with_filter_api(prompt: str):
    """
    Send the prompt to our custom safety-filter API.

    Expected response example:

    {
        "ok": True,
        "reason": "Prompt is safe"
    }

    or

    {
        "ok": False,
        "reason": "Unsafe content detected"
    }
    """

    try:

        response = requests.post(
            FILTER_API_URL,
            json={"prompt": prompt},
            timeout=10,
        )

        response.raise_for_status()

        data = response.json()

        if not isinstance(data, dict):
            return {
                "ok": False,
                "reason": "Invalid response from safety API.",
            }

        return data

    except requests.exceptions.Timeout:

        return {
            "ok": False,
            "reason": "Safety filter API timed out.",
        }

    except requests.exceptions.RequestException as e:

        return {
            "ok": False,
            "reason": f"Safety filter API error: {str(e)}",
        }

    except Exception as e:

        return {
            "ok": False,
            "reason": f"Unexpected safety-filter error: {str(e)}",
        }


# ---------------------------------------------------------
# Prompt Enhancement
# ---------------------------------------------------------

def enhance_prompt(raw_prompt: str) -> str:
    """
    Improve the original prompt using a Hugging Face
    language model.
    """

    prompt = f"""
{ENHANCE_SYS}

User prompt:
{raw_prompt}
"""

    try:

        result = generate_response(
            prompt,
            temperature=0.4,
            max_tokens=220,
        )

        if result:
            return result.strip()

    except Exception:

        # If enhancement fails, continue with
        # the original safe prompt.
        pass

    return raw_prompt.strip()


# ---------------------------------------------------------
# Image Generation
# ---------------------------------------------------------

def generate_image(prompt: str):
    """
    Generate an image using Hugging Face.

    The prompt is checked again before sending
    it to the image-generation model.
    """

    # Safety check before generation
    filter_result = check_prompt_with_filter_api(prompt)

    if not filter_result.get("ok"):

        reason = filter_result.get(
            "reason",
            "Unsafe prompt detected.",
        )

        return None, (
            "⚠️ Prompt blocked by the safety filter.\n\n"
            f"{reason}"
        )

    try:

        # First try generation with negative prompt
        image = image_client.text_to_image(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            model=MODEL_ID,
        )

        return image, None

    except Exception as e:

        error_message = str(e)

        # -------------------------------------------------
        # Some providers/models may not accept
        # negative_prompt.
        # -------------------------------------------------

        if (
            "negative_prompt" in error_message.lower()
            or "unexpected keyword" in error_message.lower()
        ):

            try:

                image = image_client.text_to_image(
                    prompt=prompt,
                    model=MODEL_ID,
                )

                return image, None

            except Exception as second_error:

                error_message = str(second_error)

        # -------------------------------------------------
        # Common Hugging Face errors
        # -------------------------------------------------

        if any(
            error in error_message
            for error in [
                "402",
                "Payment Required",
                "pre-paid credits",
            ]
        ):

            return None, (
                "❌ Hugging Face image generation requires "
                "credits, or this model is not available "
                "through the selected provider.\n\n"
                f"Raw error: {error_message}"
            )

        if (
            "404" in error_message
            or "Not Found" in error_message
        ):

            return None, (
                "❌ The selected model is not currently "
                "served through this Hugging Face provider.\n\n"
                f"Raw error: {error_message}"
            )

        if (
            "503" in error_message
            or "temporarily unavailable" in error_message.lower()
        ):

            return None, (
                "❌ The Hugging Face model is temporarily "
                "unavailable.\n\n"
                f"Raw error: {error_message}"
            )

        return None, (
            "❌ Error during image generation:\n\n"
            f"{error_message}"
        )


# ---------------------------------------------------------
# Main Streamlit Application
# ---------------------------------------------------------

def main():

    st.set_page_config(
        page_title="Safe AI Image Generator",
        page_icon="🖼️",
        layout="centered",
    )

    st.title("🖼️ Safe AI Image Generator")

    st.write(
        "Generate AI images using Hugging Face while "
        "checking prompts with a custom safety filter."
    )

    st.info(
        "🔐 Flow: Enter Prompt → Safety Check → "
        "Enhance Prompt → Safety Check Again → "
        "Generate Image → Download Image"
    )

    # -----------------------------------------------------
    # Initialize Session State
    # -----------------------------------------------------

    if "generated_image" not in st.session_state:
        st.session_state.generated_image = None

    if "final_prompt" not in st.session_state:
        st.session_state.final_prompt = ""

    if "original_prompt" not in st.session_state:
        st.session_state.original_prompt = ""

    # -----------------------------------------------------
    # Prompt Form
    # -----------------------------------------------------

    with st.form("image_form"):

        raw_prompt = st.text_area(
            "Image Description",
            height=120,
            placeholder=(
                "Example: A cozy wooden cabin in snowy "
                "mountains at sunrise with cinematic lighting"
            ),
        )

        submit = st.form_submit_button(
            "🎨 Generate Image",
            use_container_width=True,
        )

    # -----------------------------------------------------
    # Process Prompt
    # -----------------------------------------------------

    if submit:

        raw_prompt = raw_prompt.strip()

        # Empty prompt check
        if not raw_prompt:

            st.warning(
                "⚠️ Please enter an image description."
            )

            return

        # -------------------------------------------------
        # STEP 1 — Check original prompt
        # -------------------------------------------------

        with st.spinner(
            "Checking your prompt for safety..."
        ):

            original_check = (
                check_prompt_with_filter_api(
                    raw_prompt
                )
            )

        if not original_check.get("ok"):

            reason = original_check.get(
                "reason",
                "Unsafe prompt detected.",
            )

            st.error(
                f"⚠️ Prompt blocked.\n\n{reason}"
            )

            return

        st.success(
            "✅ Original prompt passed the safety check."
        )

        # -------------------------------------------------
        # STEP 2 — Enhance prompt
        # -------------------------------------------------

        with st.spinner(
            "✨ Enhancing your prompt..."
        ):

            final_prompt = enhance_prompt(
                raw_prompt
            )

        # -------------------------------------------------
        # STEP 3 — Check enhanced prompt
        # -------------------------------------------------

        with st.spinner(
            "Checking enhanced prompt..."
        ):

            enhanced_check = (
                check_prompt_with_filter_api(
                    final_prompt
                )
            )

        if not enhanced_check.get("ok"):

            reason = enhanced_check.get(
                "reason",
                "Unsafe content detected.",
            )

            st.error(
                "⚠️ The enhanced prompt was blocked.\n\n"
                f"{reason}"
            )

            return

        st.success(
            "✅ Enhanced prompt passed the safety check."
        )

        # -------------------------------------------------
        # Display Enhanced Prompt
        # -------------------------------------------------

        st.markdown("### ✨ Enhanced Prompt")

        st.code(
            final_prompt,
            language=None,
        )

        # -------------------------------------------------
        # STEP 4 — Generate Image
        # -------------------------------------------------

        with st.spinner(
            "🎨 Generating image with Hugging Face..."
        ):

            image, error = generate_image(
                final_prompt
            )

        if error:

            st.error(error)

            return

        # -------------------------------------------------
        # Save to Session State
        # -------------------------------------------------

        st.session_state.generated_image = image
        st.session_state.final_prompt = final_prompt
        st.session_state.original_prompt = raw_prompt

    # -----------------------------------------------------
    # Display Image From Session State
    # -----------------------------------------------------

    image = st.session_state.generated_image

    if image is not None:

        st.divider()

        st.markdown("### 🖼️ Generated Image")

        st.image(
            image,
            caption="AI Generated Image",
            use_container_width=True,
        )

        # -------------------------------------------------
        # Image Download
        # -------------------------------------------------

        buffer = BytesIO()

        image.save(
            buffer,
            format="PNG",
        )

        buffer.seek(0)

        st.download_button(
            label="📥 Download Image",
            data=buffer.getvalue(),
            file_name="ai_generated_image.png",
            mime="image/png",
            use_container_width=True,
        )

        # -------------------------------------------------
        # Show Prompt Details
        # -------------------------------------------------

        with st.expander(
            "View Prompt Details"
        ):

            st.markdown(
                "**Original Prompt**"
            )

            st.write(
                st.session_state.original_prompt
            )

            st.markdown(
                "**Enhanced Prompt**"
            )

            st.write(
                st.session_state.final_prompt
            )

    # -----------------------------------------------------
    # Explanation
    # -----------------------------------------------------

    with st.expander(
        "🔐 How does the safety system work?"
    ):

        st.write(
            """
            The application performs multiple safety checks.

            1. The user's original prompt is sent to the
            custom prompt-filter API.

            2. Unsafe prompts are stopped before they reach
            the AI models.

            3. Safe prompts are enhanced using a Hugging
            Face language model.

            4. The enhanced prompt is checked again because
            the enhancement model may add new words.

            5. Only prompts that pass both checks are sent
            to the Hugging Face image-generation model.

            6. The generated image is stored using
            Streamlit Session State.

            7. The image can then be downloaded as a
            PNG file.
            """
        )


# ---------------------------------------------------------
# Run Application
# ---------------------------------------------------------

if __name__ == "__main__":
    main()